# تحويل أوزان مصعب لأندرويد فقط
نوتبوك مستقل: أضف مخرجات جلسة التدريب التي تحتوي laptop_fp16.pt كـInput، فعّل الإنترنت لتنزيل مكتبات التحويل، ثم Run All. لا يحتاج GPU أو بيانات التدريب. نزّل musab_android.onnx من Output وحمّله في التطبيق. هذا لا يشغّل التدريب ولا يعدّل أوزانه.


In [ ]:
%pip install -q onnx==1.17.0 onnxruntime==1.22.0


In [ ]:
from pathlib import Path
import sys, subprocess
SOURCES = {'musab_model.py': '"""36,321,280 parameter MathCore; cached CPU/CUDA inference without a trainer import."""\nimport argparse\nimport contextlib\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom musab_data import MAX_SEQ_LEN, ANSWER_BUDGET, BOS, EOS, PAD, encode_prompt, decode, pack\n\nMODEL_CONFIG = dict(vocab=259, layers=12, dim=512, heads=8, kv_heads=2, head_dim=64, ff=1536, context=256)\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n    def forward(self, x):\n        scale = torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-5)\n        return x * scale.to(x.dtype) * self.weight\n\ndef rope(x, cos, sin):\n    a, b = x[..., ::2], x[..., 1::2]\n    c, s = cos.to(x.dtype), sin.to(x.dtype)\n    return torch.stack((a*c-b*s, a*s+b*c), -1).flatten(-2)\n\nclass Attention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        d, h = cfg[\'dim\'], cfg[\'head_dim\']\n        self.q = nn.Linear(d, cfg[\'heads\'] * h, bias=False)\n        self.k = nn.Linear(d, cfg[\'kv_heads\'] * h, bias=False)\n        self.v = nn.Linear(d, cfg[\'kv_heads\'] * h, bias=False)\n        self.o = nn.Linear(cfg[\'heads\'] * h, d, bias=False)\n    def forward(self, x, cos, sin, cache=None, use_cache=False):\n        b, t, _ = x.shape\n        c = self.cfg\n        q = self.q(x).view(b, t, c[\'heads\'], c[\'head_dim\']).transpose(1, 2)\n        k = self.k(x).view(b, t, c[\'kv_heads\'], c[\'head_dim\']).transpose(1, 2)\n        v = self.v(x).view(b, t, c[\'kv_heads\'], c[\'head_dim\']).transpose(1, 2)\n        q, k = rope(q, cos, sin), rope(k, cos, sin)\n        if cache is not None:\n            if t != 1:\n                raise ValueError(\'Cached continuation accepts one token per sequence\')\n            k, v = torch.cat((cache[0], k), -2), torch.cat((cache[1], v), -2)\n        new_cache = (k, v) if use_cache else None\n        rep = c[\'heads\'] // c[\'kv_heads\']\n        y = F.scaled_dot_product_attention(q, k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1),\n                                          is_causal=(cache is None))\n        return self.o(y.transpose(1, 2).contiguous().view(b, t, c[\'dim\'])), new_cache\n\nclass FeedForward(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.g = nn.Linear(cfg[\'dim\'], cfg[\'ff\'], bias=False)\n        self.u = nn.Linear(cfg[\'dim\'], cfg[\'ff\'], bias=False)\n        self.d = nn.Linear(cfg[\'ff\'], cfg[\'dim\'], bias=False)\n    def forward(self, x):\n        return self.d(F.silu(self.g(x)) * self.u(x))\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.n1, self.n2 = RMSNorm(cfg[\'dim\']), RMSNorm(cfg[\'dim\'])\n        self.attn, self.ff = Attention(cfg), FeedForward(cfg)\n    def forward(self, x, c, s, cache=None, use_cache=False):\n        y, new_cache = self.attn(self.n1(x), c, s, cache, use_cache)\n        x = x + y\n        return x + self.ff(self.n2(x)), new_cache\n\nclass MathCore(nn.Module):\n    def __init__(self, cfg=None):\n        super().__init__()\n        self.cfg = dict(MODEL_CONFIG if cfg is None else cfg)\n        c = self.cfg\n        self.embed = nn.Embedding(c[\'vocab\'], c[\'dim\'])\n        self.blocks = nn.ModuleList([Block(c) for _ in range(c[\'layers\'])])\n        self.norm = RMSNorm(c[\'dim\'])\n        inv = 1.0 / (10000.0 ** (torch.arange(0, c[\'head_dim\'], 2).float() / c[\'head_dim\']))\n        angles = torch.outer(torch.arange(c[\'context\']).float(), inv)[None, None]\n        self.register_buffer(\'rope_cos\', angles.cos(), persistent=False)\n        self.register_buffer(\'rope_sin\', angles.sin(), persistent=False)\n        self.apply(self._init)\n        for block in self.blocks:\n            nn.init.normal_(block.attn.o.weight, std=0.02 / math.sqrt(2 * c[\'layers\']))\n            nn.init.normal_(block.ff.d.weight, std=0.02 / math.sqrt(2 * c[\'layers\']))\n    @staticmethod\n    def _init(m):\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, std=0.02)\n    def forward(self, ids, caches=None, use_cache=False):\n        offset = 0 if caches is None else caches[0][0].shape[-2]\n        end = offset + ids.shape[1]\n        if end > self.cfg[\'context\']:\n            raise ValueError(\'Context exhausted\')\n        c, s = self.rope_cos[..., offset:end, :], self.rope_sin[..., offset:end, :]\n        x, result = self.embed(ids), []\n        for i, block in enumerate(self.blocks):\n            x, cache = block(x, c, s, None if caches is None else caches[i], use_cache)\n            if use_cache:\n                result.append(cache)\n        # Inference computes logits only for the token being predicted.\n        hidden = self.norm(x[:, -1:] if use_cache else x)\n        logits = F.linear(hidden, self.embed.weight)\n        return (logits, result) if use_cache else logits\n\ndef collate(pairs, device):\n    # None is padding of the FINAL global batch, with exactly zero supervised tokens.\n    items = [pack(*r) if r is not None else ([BOS, EOS], [-100, -100]) for r in pairs]\n    length = max(len(x) for x, _ in items)\n    x = torch.full((len(items), length), PAD, dtype=torch.long)\n    y = torch.full_like(x, -100)\n    for i, (ids, labels) in enumerate(items):\n        x[i, :len(ids)] = torch.tensor(ids)\n        y[i, :len(labels)] = torch.tensor(labels)\n    return x.to(device), y.to(device)\n\ndef loss_stats(logits, labels):\n    # Explicit FP32 CE also protects evaluation outside autocast.\n    z, y = logits[:, :-1].float(), labels[:, 1:]\n    loss = F.cross_entropy(z.reshape(-1, z.shape[-1]), y.reshape(-1), ignore_index=-100, reduction=\'sum\')\n    good = y != -100\n    return loss, good.sum(), ((z.argmax(-1) == y) & good).sum()\n\n@torch.inference_mode()\ndef generate_batch(model, questions, device, max_new=ANSWER_BUDGET):\n    """Group equal-length prompts: no padding positions or attention-mask ambiguity."""\n    model.eval()\n    groups, result = {}, [None] * len(questions)\n    for i, q in enumerate(questions):\n        p = encode_prompt(q)\n        groups.setdefault(len(p), []).append((i, p))\n    for length, entries in groups.items():\n        x = torch.tensor([p for _, p in entries], device=device)\n        tokens = [[] for _ in entries]\n        ended = [False] * len(entries)\n        caches = None\n        for _ in range(min(max_new, model.cfg[\'context\'] - length)):\n            amp = torch.autocast(\'cuda\', dtype=torch.float16) if device.type == \'cuda\' else contextlib.nullcontext()\n            with amp:\n                logits, caches = model(x, caches=caches, use_cache=True)\n            # PAD/BOS cannot form a valid answer. EOS remains legal.\n            logits[:, -1, PAD] = -float(\'inf\')\n            logits[:, -1, BOS] = -float(\'inf\')\n            next_ids = logits[:, -1].argmax(-1).tolist()\n            for j, token in enumerate(next_ids):\n                if not ended[j]:\n                    if token == EOS:\n                        ended[j] = True\n                    else:\n                        tokens[j].append(token)\n            if all(ended):\n                break\n            x = torch.tensor([[EOS if ended[j] else token] for j, token in enumerate(next_ids)], device=device)\n        for j, (i, _) in enumerate(entries):\n            result[i] = dict(answer=decode(tokens[j]).strip(), ended=ended[j], tokens=len(tokens[j]))\n    return result\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--weights\', required=True)\n    p.add_argument(\'--question\', required=True)\n    p.add_argument(\'--threads\', type=int, default=4)\n    p.add_argument(\'--int8\', action=\'store_true\', help=\'Optional dynamic CPU quantization; can change answers\')\n    a = p.parse_args()\n    torch.set_num_threads(max(1, a.threads))\n    ck = torch.load(a.weights, map_location=\'cpu\', weights_only=True)\n    if ck.get(\'format\') != \'MUSAB_V12_INFERENCE\':\n        raise ValueError(\'Use laptop_fp16.pt, not the training resume checkpoint\')\n    model = MathCore(ck[\'model_config\'])\n    model.load_state_dict(ck[\'model\'], strict=True)\n    del ck\n    model.eval()\n    if a.int8:\n        model = torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)\n    r = generate_batch(model, [a.question], torch.device(\'cpu\'))[0]\n    print(r[\'answer\'])\n    if not r[\'ended\']:\n        print(\'[Generation reached its length limit without EOS]\')\n\nif __name__ == \'__main__\':\n    main()\n', 'musab_data.py': "PAD, BOS, EOS, BYTE_OFFSET = 0, 1, 2, 3\nMAX_SEQ_LEN, PROMPT_BUDGET, ANSWER_BUDGET = 256, 192, 64\n\ndef encode_prompt(q):\n    ids = [BOS] + [b + BYTE_OFFSET for b in ('Question: ' + q + '\\nAnswer: ').encode('utf-8')]\n    if len(ids) > PROMPT_BUDGET:\n        raise ValueError('Question exceeds prompt budget')\n    return ids\n\ndef decode(ids):\n    return bytes(i-BYTE_OFFSET for i in ids if BYTE_OFFSET <= i < 259).decode('utf-8', errors='replace')\n\ndef pack(q,a):\n    raise RuntimeError('Training is not part of this Android project')\n", 'export_android.py': '"""Convert a COPY of laptop_fp16.pt to standalone Android ONNX. Never trains."""\nimport argparse\nimport tempfile\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nimport onnx\nimport onnxruntime as ort\nfrom musab_model import MathCore, MODEL_CONFIG\nfrom musab_data import encode_prompt\n\nclass LastLogits(nn.Module):\n    def __init__(self, model):\n        super().__init__(); self.model=model\n    def forward(self, input_ids):\n        return self.model(input_ids)[:, -1, :]\n\ndef export(model, destination):\n    model=model.float().cpu().eval()\n    wrapper=LastLogits(model).eval()\n    destination=Path(destination); destination.parent.mkdir(parents=True,exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=destination.parent) as td:\n        temp=Path(td)/\'model.onnx\'\n        with torch.inference_mode():\n            torch.onnx.export(wrapper, (torch.tensor([encode_prompt(\'1+1\')]),), str(temp),\n                input_names=[\'input_ids\'], output_names=[\'logits\'], opset_version=17,\n                dynamic_axes={\'input_ids\': {1:\'sequence\'}}, dynamo=False,\n                external_data=False)\n        graph=onnx.load(str(temp))\n        onnx.helper.set_model_props(graph, {\'musab_format\':\'musab-v12-full-v1\',\n            \'tokenizer\':\'utf8-byte-offset-3\', \'context\':\'256\', \'answer_budget\':\'64\'})\n        onnx.checker.check_model(graph)\n        onnx.save(graph,str(temp))\n        session=ort.InferenceSession(str(temp),providers=[\'CPUExecutionProvider\'])\n        # Compare logits at multiple sequence lengths, including the context boundary.\n        for length in (20,37,191,255):\n            ids=torch.tensor([[1]+[3+(i*17)%256 for i in range(length-1)]])\n            with torch.inference_mode(): expected=wrapper(ids).numpy()\n            actual=session.run(None,{\'input_ids\':ids.numpy()})[0]\n            np.testing.assert_allclose(actual,expected,rtol=2e-3,atol=2e-4)\n        # Compare several consecutive autoregressive decisions against PyTorch.\n        ids=torch.tensor([encode_prompt(\'What is 12 + 7?\')])\n        for _ in range(8):\n            with torch.inference_mode(): a=wrapper(ids).numpy()\n            b=session.run(None,{\'input_ids\':ids.numpy()})[0]\n            a[:,:2]=-np.inf;b[:,:2]=-np.inf\n            if int(a.argmax())!=int(b.argmax()): raise ValueError(\'Export changed greedy token decisions\')\n            token=int(a.argmax())\n            if token==2:break\n            ids=torch.cat([ids,torch.tensor([[token]])],dim=1)\n        del session\n        temp.replace(destination)\n    return destination\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--weights\',required=True,help=\'laptop_fp16.pt exported by V12.2\')\n    parser.add_argument(\'--output\',default=\'musab_android.onnx\')\n    args=parser.parse_args()\n    if Path(args.weights).resolve()==Path(args.output).resolve():raise ValueError(\'Input and output must differ\')\n    torch.set_num_threads(2)\n    ck=torch.load(args.weights,map_location=\'cpu\',weights_only=True)\n    if ck.get(\'format\')!=\'MUSAB_V12_INFERENCE\' or ck.get(\'model_config\')!=MODEL_CONFIG:\n        raise ValueError(\'Use V12.2 laptop_fp16.pt; resume.pt is training state, not an inference export\')\n    model=MathCore();model.load_state_dict(ck[\'model\'],strict=True);del ck\n    print(\'EXPORT_VERIFIED:\', export(model,args.output))\nif __name__==\'__main__\':main()\n'}
work=Path('/kaggle/working/android_export_tools')
work.mkdir(exist_ok=True)
for name,content in SOURCES.items(): (work/name).write_text(content)
weights=sorted(Path('/kaggle/input').rglob('laptop_fp16.pt'))
if len(weights)!=1: raise RuntimeError(f'أضف مخرجات جلسة واحدة فقط؛ وجدت {len(weights)} ملفات أوزان')
print('Selected:',weights[0])
subprocess.run([sys.executable,str(work/'export_android.py'),'--weights',str(weights[0]),'--output','/kaggle/working/musab_android.onnx'],check=True)
